In [ ]:
""" FULLY CONNECTED NEURAL NETWORK TRAINING"""
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64, shuffle = True, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle = True, num_workers=2, pin_memory=True)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

learning_rate = ((20)*1e-3)
batch_size = 64
epochs = 25

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

torch.save(model, 'Build1.pth')

100%|██████████| 26.4M/26.4M [00:00<00:00, 116MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.68MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 59.9MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 6.63MB/s]


Epoch 1
-------------------------------
loss: 2.303107  [   64/60000]
loss: 1.901254  [ 6464/60000]
loss: 1.245172  [12864/60000]
loss: 0.922867  [19264/60000]
loss: 0.792944  [25664/60000]
loss: 0.775485  [32064/60000]
loss: 0.805797  [38464/60000]
loss: 0.610210  [44864/60000]
loss: 0.681496  [51264/60000]
loss: 0.808468  [57664/60000]
Test Error: 
 Accuracy: 78.0%, Avg loss: 0.641409 

Epoch 2
-------------------------------
loss: 0.640706  [   64/60000]
loss: 0.470388  [ 6464/60000]
loss: 0.602096  [12864/60000]
loss: 0.454277  [19264/60000]
loss: 0.475068  [25664/60000]
loss: 0.565642  [32064/60000]
loss: 0.347288  [38464/60000]
loss: 0.464515  [44864/60000]
loss: 0.522623  [51264/60000]
loss: 0.597380  [57664/60000]
Test Error: 
 Accuracy: 80.7%, Avg loss: 0.539383 

Epoch 3
-------------------------------
loss: 0.381947  [   64/60000]
loss: 0.564932  [ 6464/60000]
loss: 0.438342  [12864/60000]
loss: 0.378693  [19264/60000]
loss: 0.427803  [25664/60000]
loss: 0.687558  [32064/600

KeyboardInterrupt: 

In [ ]:
""" CONVOLUTIONAL NEURAL NETWORK TRAINING"""
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64, shuffle = True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle = True)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution_relu_stack = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2),
            nn.Flatten(),
            nn.Linear(7*7*128, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        logits = self.convolution_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
model = model.to(device)

learning_rate = ((20)*1e-3)
batch_size = 64
epochs = 60

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

torch.save(model, 'Build2.pth')

Epoch 1
-------------------------------
loss: 2.302944  [   64/60000]
loss: 2.296549  [ 6464/60000]
loss: 2.280592  [12864/60000]
loss: 1.367518  [19264/60000]
loss: 0.982494  [25664/60000]
loss: 1.126020  [32064/60000]
loss: 0.887543  [38464/60000]
loss: 0.923982  [44864/60000]
loss: 0.690189  [51264/60000]
loss: 0.405502  [57664/60000]
Test Error: 
 Accuracy: 66.5%, Avg loss: 0.892340 

Epoch 2
-------------------------------
loss: 0.817865  [   64/60000]
loss: 0.775984  [ 6464/60000]
loss: 0.702153  [12864/60000]
loss: 0.496620  [19264/60000]
loss: 0.789873  [25664/60000]
loss: 0.653894  [32064/60000]
loss: 0.582579  [38464/60000]
loss: 0.484928  [44864/60000]
loss: 0.469183  [51264/60000]
loss: 0.725748  [57664/60000]
Test Error: 
 Accuracy: 77.5%, Avg loss: 0.572098 

Epoch 3
-------------------------------
loss: 0.459188  [   64/60000]
loss: 0.326996  [ 6464/60000]
loss: 0.574880  [12864/60000]
loss: 0.523245  [19264/60000]
loss: 0.775689  [25664/60000]
loss: 0.469279  [32064/600

In [ ]:
""" FULLY CONNECTED NEURAL NETWORK PREDICTION"""
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path
import csv
from PIL import Image

device = torch.device("cpu")

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_dataloader = DataLoader(test_data, batch_size=100, shuffle = True)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = torch.load("Build1.pth", weights_only= False)
model = model.to(device)
model.eval()

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

loss_fn = nn.CrossEntropyLoss()
t = []
for i in range(100):
  if i < 10:
    t.append("00"+str(i)+".png")
    continue
  t.append("0"+str(i)+".png")
with open("Build1 Predictions.csv", "w") as file:
    file.write("image_id,label\n")
for i in t:
  x = Path("Images/"+i)
  img = Image.open(x)
  img_tensor = v2.ToTensor()(img)
  img_tensor = img_tensor.unsqueeze(0)

  with torch.no_grad():
    pred = model(img_tensor)

  prediction = pred.argmax(1).item()

  with open("Build1 Predictions.csv", "a") as file:
    file.write(i[:3]+","+str(prediction)+"\n")






for t in range(1):
    test_loop(test_dataloader, model, loss_fn)
print("Done!")


/usr/local/lib/python3.12/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Test Error: 
 Accuracy: 87.8%, Avg loss: 0.340371 

Done!


In [ ]:
""" CONVOLUTIONAL NEURAL NETWORK PREDICTION"""
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path
import csv
from PIL import Image

device = torch.device("cpu")

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_dataloader = DataLoader(test_data, batch_size=100, shuffle = True)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution_relu_stack = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2),
            nn.Flatten(),
            nn.Linear(7*7*128, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        logits = self.convolution_relu_stack(x)
        return logits

model = torch.load("Build2.pth", weights_only= False)
model = model.to(device)
model.eval()

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

loss_fn = nn.CrossEntropyLoss()
t = []
for i in range(100):
  if i < 10:
    t.append("00"+str(i)+".png")
    continue
  t.append("0"+str(i)+".png")
with open("Build2 Predictions.csv", "w") as file:
    file.write("image_id,label\n")
for i in t:
  x = Path("Images/"+i)
  img = Image.open(x)
  img_tensor = v2.ToTensor()(img)
  img_tensor = img_tensor.unsqueeze(0)

  with torch.no_grad():
    pred = model(img_tensor)

  prediction = pred.argmax(1).item()

  with open("Build2 Predictions.csv", "a") as file:
    file.write(i[:3]+","+str(prediction)+"\n")






for t in range(1):
    test_loop(test_dataloader, model, loss_fn)
print("Done!")


Test Error: 
 Accuracy: 91.9%, Avg loss: 0.715182 

Done!
